# 🎙️ Moshiko — Full Setup, Test & Benchmark Notebook
**University Research Project**

---

## 📋 What This Notebook Does
| Step | Description |
|------|-------------|
| **Cell 1** | Mount Google Drive & create folder structure |
| **Cell 2** | Check GPU & environment |
| **Cell 3** | Install dependencies |
| **Cell 4** | Download Moshiko models (with checkpoint system) |
| **Cell 5** | Load models into memory |
| **Cell 6** | TEST 1 — Basic audio input → speech output |
| **Cell 7** | TEST 2 — Measure latency & speed |
| **Cell 8** | TEST 3 — Compare bf16 vs q8 quality |
| **Cell 9** | Save all outputs to Google Drive |

---

## ⚠️ IMPORTANT — Read Before Running
- **Runtime → Change runtime type → T4 GPU** must be selected before running
- Run cells **in order**, top to bottom
- If session disconnects: re-run Cell 1, Cell 2, Cell 3, then jump to Cell 4 (it will skip already-downloaded files)
- All outputs are auto-saved to your shared Google Drive

---
**Group Members:** _(add your names here)_  
**Last Updated:** _(auto-updated when you run Cell 1)_

---
## 📁 CELL 1 — Mount Google Drive & Create Folder Structure

In [ ]:
# ============================================================
# CELL 1 — Google Drive Mount & Folder Structure
# ============================================================
# PURPOSE: Connect to shared Google Drive and create all
#          project folders automatically.
#
# ✅ Safe to re-run: will NOT delete existing files
# ============================================================

import os
from datetime import datetime
from google.colab import drive

# --- Step 1: Mount Google Drive ---
print("📂 Mounting Google Drive...")
drive.mount('/content/drive', force_remount=False)
print("✅ Google Drive mounted successfully!\n")

# ============================================================
# FOLDER STRUCTURE DEFINITION
# All project files live under: MyDrive/Moshiko_Project/
# ============================================================
BASE_DIR = '/content/drive/MyDrive/Moshiko_Project'

FOLDERS = {
    'base'        : BASE_DIR,
    'models'      : f'{BASE_DIR}/models',           # Downloaded model weights
    'models_bf16' : f'{BASE_DIR}/models/bf16',      # Full precision weights
    'models_q8'   : f'{BASE_DIR}/models/q8',        # INT8 quantized weights
    'checkpoints' : f'{BASE_DIR}/checkpoints',      # Session resume points
    'outputs'     : f'{BASE_DIR}/outputs',           # All test results
    'audio_in'    : f'{BASE_DIR}/outputs/audio_input',   # Input audio files
    'audio_out'   : f'{BASE_DIR}/outputs/audio_output',  # Generated audio
    'benchmarks'  : f'{BASE_DIR}/outputs/benchmarks',    # Latency/speed results
    'comparisons' : f'{BASE_DIR}/outputs/comparisons',   # bf16 vs q8 results
    'logs'        : f'{BASE_DIR}/logs',             # Session logs
    'notebooks'   : f'{BASE_DIR}/notebooks',        # Notebook backups
}

# --- Step 2: Create all folders ---
print("📁 Creating project folder structure...")
for name, path in FOLDERS.items():
    os.makedirs(path, exist_ok=True)
    print(f"   {'✅ Exists' if os.path.exists(path) else '🆕 Created'}: {path.replace(BASE_DIR, 'Moshiko_Project')}")

# --- Step 3: Write a session log ---
session_time = datetime.now().strftime('%Y-%m-%d %H:%M:%S')
log_file = f"{FOLDERS['logs']}/session_log.txt"
with open(log_file, 'a') as f:
    f.write(f"\n[{session_time}] New session started in Google Colab")

# --- Step 4: Update notebook timestamp in README ---
readme_path = f"{BASE_DIR}/README.txt"
readme_content = f"""╔══════════════════════════════════════════════════════════╗
║           MOSHIKO RESEARCH PROJECT — README              ║
╠══════════════════════════════════════════════════════════╣
║  Last Session : {session_time}               ║
╠══════════════════════════════════════════════════════════╣
║  FOLDER STRUCTURE                                        ║
║                                                          ║
║  Moshiko_Project/                                        ║
║  ├── models/                                             ║
║  │   ├── bf16/     ← Full precision model weights        ║
║  │   └── q8/       ← INT8 quantized model weights        ║
║  ├── checkpoints/  ← Session resume points               ║
║  ├── outputs/                                            ║
║  │   ├── audio_input/   ← Your test audio files          ║
║  │   ├── audio_output/  ← Generated speech outputs       ║
║  │   ├── benchmarks/    ← Latency & speed results        ║
║  │   └── comparisons/   ← bf16 vs q8 quality tests       ║
║  ├── logs/         ← Session activity logs               ║
║  └── notebooks/    ← Saved notebook backups              ║
╠══════════════════════════════════════════════════════════╣
║  HOW TO RESUME AFTER DISCONNECT:                         ║
║  1. Re-run Cell 1 (Drive mount)                          ║
║  2. Re-run Cell 2 (GPU check)                            ║
║  3. Re-run Cell 3 (Install packages)                     ║
║  4. Run Cell 4 (will auto-skip downloaded files)         ║
║  5. Re-run Cell 5 (Load model)                           ║
║  6. Run any test cell you need                           ║
╚══════════════════════════════════════════════════════════╝
"""
with open(readme_path, 'w') as f:
    f.write(readme_content)

print(f"""
╔══════════════════════════════════════════════╗
║  ✅ CELL 1 COMPLETE                          ║
║  📁 Project root: MyDrive/Moshiko_Project/   ║
║  📝 README.txt created/updated               ║
║  📋 Session logged at {session_time[:10]}           ║
╚══════════════════════════════════════════════╝
""")

# Make FOLDERS accessible to all other cells
import builtins
builtins.FOLDERS = FOLDERS
builtins.BASE_DIR = BASE_DIR
print("➡️  Ready for Cell 2")

---
## 🖥️ CELL 2 — Check GPU & Environment

In [ ]:
# ============================================================
# CELL 2 — GPU & Environment Check
# ============================================================
# PURPOSE: Verify T4 GPU is available and has enough VRAM.
#          Automatically selects the right model precision.
#
# ✅ Safe to re-run anytime
# ============================================================

import subprocess
import sys
import json
import os

print("🔍 Checking environment...\n")

# --- GPU Check ---
try:
    import torch
    gpu_available = torch.cuda.is_available()
    if gpu_available:
        gpu_name = torch.cuda.get_device_name(0)
        vram_total = torch.cuda.get_device_properties(0).total_memory / 1e9
        vram_free  = (torch.cuda.get_device_properties(0).total_memory 
                      - torch.cuda.memory_allocated(0)) / 1e9
        print(f"  GPU      : {gpu_name}")
        print(f"  VRAM     : {vram_total:.1f} GB total | {vram_free:.1f} GB free")
    else:
        print("  ❌ NO GPU DETECTED!")
        print("  ⚠️  Go to: Runtime → Change runtime type → T4 GPU")
        raise SystemExit("Please enable GPU and re-run.")
except ImportError:
    gpu_name  = "Unknown (torch not installed yet)"
    vram_total = 0
    vram_free  = 0
    print("  ⚠️  PyTorch not installed yet — will check after Cell 3")

# --- Python Version ---
py_version = sys.version.split()[0]
print(f"  Python   : {py_version}")
py_ok = tuple(int(x) for x in py_version.split('.')[:2]) >= (3, 10)
print(f"  Python OK: {'✅ Yes' if py_ok else '❌ Needs 3.10+'}")

# --- Decide which model to use based on VRAM ---
if vram_total > 0:
    if vram_total >= 35:
        MODEL_PRECISION = 'bf16'
        MODEL_REPO      = 'kyutai/moshiko-pytorch-bf16'
        precision_note  = 'Full precision (A100 detected)'
    elif vram_total >= 14:
        MODEL_PRECISION = 'q8'
        MODEL_REPO      = 'kyutai/moshiko-pytorch-q8'
        precision_note  = 'INT8 quantized (T4 safe — uses ~14GB)'
    else:
        MODEL_PRECISION = 'q8'
        MODEL_REPO      = 'kyutai/moshiko-pytorch-q8'
        precision_note  = '⚠️ Low VRAM — using INT8, may be tight'
else:
    MODEL_PRECISION = 'q8'
    MODEL_REPO      = 'kyutai/moshiko-pytorch-q8'
    precision_note  = 'Default (torch not yet installed)'

# Save config for later cells
builtins.MODEL_PRECISION = MODEL_PRECISION
builtins.MODEL_REPO      = MODEL_REPO

# --- Save environment info to Drive ---
try:
    env_info = {
        'session_time'     : datetime.now().strftime('%Y-%m-%d %H:%M:%S'),
        'gpu_name'         : gpu_name,
        'vram_total_gb'    : round(vram_total, 2),
        'vram_free_gb'     : round(vram_free,  2),
        'python_version'   : py_version,
        'selected_model'   : MODEL_REPO,
        'model_precision'  : MODEL_PRECISION,
    }
    env_path = f"{FOLDERS['logs']}/environment_info.json"
    with open(env_path, 'w') as f:
        json.dump(env_info, f, indent=2)
    print(f"\n  💾 Environment info saved to Drive")
except Exception as e:
    print(f"  ⚠️  Could not save env info: {e} (run Cell 1 first)")

print(f"""
╔══════════════════════════════════════════════════════╗
║  ✅ CELL 2 COMPLETE                                  ║
║  🎯 Selected model : {MODEL_REPO:<32}║
║  📊 Precision      : {MODEL_PRECISION:<32}║
║  📝 Note           : {precision_note:<32}║
╚══════════════════════════════════════════════════════╝
""")
print("➡️  Ready for Cell 3")

---
## 📦 CELL 3 — Install All Dependencies

In [ ]:
# ============================================================
# CELL 3 — Install Dependencies
# ============================================================
# PURPOSE: Install moshi, huggingface_hub, torchaudio, and
#          all required packages.
#
# ⏱️  Takes ~2-3 minutes on first run
# ✅ Safe to re-run (pip skips already-installed packages)
# ============================================================

import subprocess, sys, importlib

def install(package, import_name=None):
    """Install a package and verify it imported correctly."""
    name = import_name or package.split('==')[0].replace('-','_')
    try:
        importlib.import_module(name)
        print(f"  ✅ Already installed : {package}")
    except ImportError:
        print(f"  📥 Installing        : {package} ...", end='', flush=True)
        result = subprocess.run(
            [sys.executable, '-m', 'pip', 'install', '-q', package],
            capture_output=True, text=True
        )
        if result.returncode == 0:
            print(" Done ✅")
        else:
            print(f" FAILED ❌\n    Error: {result.stderr[:200]}")

print("📦 Installing required packages...\n")

packages = [
    ('moshi',          'moshi'),
    ('huggingface_hub','huggingface_hub'),
    ('torchaudio',     'torchaudio'),
    ('soundfile',      'soundfile'),
    ('numpy',          'numpy'),
    ('scipy',          'scipy'),
    ('matplotlib',     'matplotlib'),
    ('IPython',        'IPython'),
    ('tqdm',           'tqdm'),
]

for pkg, imp in packages:
    install(pkg, imp)

# --- Verify critical imports ---
print("\n🔍 Verifying critical imports...")
critical = ['torch', 'moshi', 'huggingface_hub', 'torchaudio', 'soundfile']
all_ok = True
for mod in critical:
    try:
        m = importlib.import_module(mod)
        ver = getattr(m, '__version__', 'unknown')
        print(f"  ✅ {mod:<20} v{ver}")
    except ImportError:
        print(f"  ❌ {mod:<20} IMPORT FAILED — re-run this cell")
        all_ok = False

if all_ok:
    print("""
╔══════════════════════════════════════════╗
║  ✅ CELL 3 COMPLETE — All packages OK   ║
╚══════════════════════════════════════════╝
""")
    print("➡️  Ready for Cell 4")
else:
    print("""
╔══════════════════════════════════════════╗
║  ❌ Some packages failed — re-run cell  ║
╚══════════════════════════════════════════╝
""")

---
## ⬇️ CELL 4 — Download Models (with Checkpoint System)

In [ ]:
# ============================================================
# CELL 4 — Download Models with Checkpoint System
# ============================================================
# PURPOSE: Download Moshiko model weights to Google Drive.
#          Uses a checkpoint file to track what's already
#          downloaded — so disconnects don't restart downloads.
#
# ⏱️  First run: ~10-20 min (large model files)
# ✅ Resume-safe: Already downloaded files are SKIPPED
# ============================================================

import os, json, time
from datetime import datetime
from huggingface_hub import hf_hub_download
from tqdm import tqdm

# ---- Checkpoint helpers ----
CHECKPOINT_FILE = f"{FOLDERS['checkpoints']}/download_checkpoint.json"

def load_checkpoint():
    """Load which files have already been downloaded."""
    if os.path.exists(CHECKPOINT_FILE):
        with open(CHECKPOINT_FILE, 'r') as f:
            return json.load(f)
    return {'downloaded': {}, 'last_updated': None}

def save_checkpoint(ckpt):
    """Save progress after each file download."""
    ckpt['last_updated'] = datetime.now().strftime('%Y-%m-%d %H:%M:%S')
    with open(CHECKPOINT_FILE, 'w') as f:
        json.dump(ckpt, f, indent=2)

def download_file(repo_id, filename, local_dir, ckpt, label):
    """
    Download a single file from HuggingFace Hub.
    Skips if already downloaded (checked via checkpoint).
    """
    key = f"{repo_id}/{filename}"
    local_path = os.path.join(local_dir, filename)

    # Check checkpoint
    if key in ckpt['downloaded'] and os.path.exists(local_path):
        size_mb = os.path.getsize(local_path) / 1e6
        print(f"  ⏭️  SKIP (already downloaded): {label} ({size_mb:.0f} MB)")
        return local_path

    print(f"  📥 Downloading: {label} ...", flush=True)
    start = time.time()
    try:
        path = hf_hub_download(
            repo_id=repo_id,
            filename=filename,
            local_dir=local_dir,
            local_dir_use_symlinks=False,
        )
        elapsed  = time.time() - start
        size_mb  = os.path.getsize(path) / 1e6
        print(f"     ✅ Done — {size_mb:.0f} MB in {elapsed:.0f}s")

        # Save to checkpoint
        ckpt['downloaded'][key] = {
            'path': path,
            'size_mb': round(size_mb, 1),
            'downloaded_at': datetime.now().strftime('%Y-%m-%d %H:%M:%S')
        }
        save_checkpoint(ckpt)
        return path
    except Exception as e:
        print(f"     ❌ FAILED: {e}")
        print(f"     💡 Re-run this cell to retry.")
        return None

# ---- Load existing checkpoint ----
ckpt = load_checkpoint()
if ckpt['last_updated']:
    print(f"📋 Checkpoint found — last updated: {ckpt['last_updated']}")
    print(f"   Already downloaded: {len(ckpt['downloaded'])} file(s)\n")
else:
    print("📋 No checkpoint found — starting fresh download\n")

# ---- Files to download ----
# We download BOTH q8 and bf16 Mimi weights + Moshiko weights
# q8 = for running on T4 | bf16 = for quality comparison test
files_to_download = [
    # Q8 model (primary — fits T4)
    ('kyutai/moshiko-pytorch-q8', 'mimi_weight.pt',  FOLDERS['models_q8'],  'Mimi Codec (q8)'),
    ('kyutai/moshiko-pytorch-q8', 'moshiko-q8.safetensors', FOLDERS['models_q8'], 'Moshiko LM (q8)'),
    # BF16 Mimi only (for comparison — Mimi is small ~200MB)
    ('kyutai/moshiko-pytorch-bf16', 'mimi_weight.pt', FOLDERS['models_bf16'], 'Mimi Codec (bf16)'),
]

print("⬇️  Starting downloads...\n")
downloaded_paths = {}
for repo, fname, local_dir, label in files_to_download:
    path = download_file(repo, fname, local_dir, ckpt, label)
    if path:
        downloaded_paths[label] = path

# ---- Summary ----
total_files = len(ckpt['downloaded'])
print(f"""
╔══════════════════════════════════════════════════════════╗
║  ✅ CELL 4 COMPLETE                                      ║
║  📦 Total files in Drive : {total_files} file(s)                    ║
║  💾 Checkpoint saved     : {CHECKPOINT_FILE.split('/')[-1]:<26}  ║
║                                                          ║
║  If this cell was interrupted:                           ║
║  → Re-run Cell 4 — it will resume where it stopped      ║
╚══════════════════════════════════════════════════════════╝
""")
print("➡️  Ready for Cell 5")

---
## 🧠 CELL 5 — Load Moshiko Model Into Memory

In [ ]:
# ============================================================
# CELL 5 — Load Moshiko Model
# ============================================================
# PURPOSE: Load the Moshiko q8 model from Drive into GPU memory.
#          Also loads Mimi codec (used for audio encoding/decoding).
#
# ⏱️  Takes ~1-3 minutes
# ✅ Must re-run after every session disconnect
# ============================================================

import torch, time, os
from moshi.models import loaders

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"🖥️  Using device: {DEVICE}")
if DEVICE == 'cpu':
    print("  ⚠️  WARNING: CPU mode will be very slow! Enable GPU in Runtime settings.")

# ---- Locate model files ----
Q8_DIR   = FOLDERS['models_q8']
BF16_DIR = FOLDERS['models_bf16']

mimi_q8_path   = os.path.join(Q8_DIR,   'mimi_weight.pt')
moshi_q8_path  = os.path.join(Q8_DIR,   'moshiko-q8.safetensors')
mimi_bf16_path = os.path.join(BF16_DIR, 'mimi_weight.pt')

# Check files exist
for label, path in [('Mimi q8', mimi_q8_path), ('Moshiko q8', moshi_q8_path)]:
    if not os.path.exists(path):
        print(f"  ❌ Missing: {label} at {path}")
        print("  → Please re-run Cell 4 to download missing files.")
        raise FileNotFoundError(f"Missing model file: {path}")
    print(f"  ✅ Found: {label} ({os.path.getsize(path)/1e6:.0f} MB)")

# ---- Load Mimi (Audio Codec) ----
print("\n⏳ Loading Mimi audio codec (q8)...")
start = time.time()
mimi = loaders.get_mimi(mimi_q8_path, device=DEVICE)
mimi.set_num_codebooks(8)   # Moshi uses 8 codebooks
mimi.eval()
print(f"  ✅ Mimi loaded in {time.time()-start:.1f}s")

# ---- Load Moshiko LM ----
print("\n⏳ Loading Moshiko language model (q8)...")
start = time.time()
moshi_lm = loaders.get_moshi(
    moshi_q8_path,
    device=DEVICE
)
moshi_lm.eval()
print(f"  ✅ Moshiko LM loaded in {time.time()-start:.1f}s")

# ---- VRAM usage after loading ----
if DEVICE == 'cuda':
    allocated = torch.cuda.memory_allocated() / 1e9
    reserved  = torch.cuda.memory_reserved()  / 1e9
    print(f"\n  📊 VRAM used    : {allocated:.1f} GB")
    print(f"  📊 VRAM reserved: {reserved:.1f} GB")

# Make models globally accessible
builtins.mimi     = mimi
builtins.moshi_lm = moshi_lm
builtins.DEVICE   = DEVICE

print(f"""
╔══════════════════════════════════════════════════╗
║  ✅ CELL 5 COMPLETE — Models loaded!             ║
║  🎵 Mimi codec   : Ready (audio tokenizer)       ║
║  🧠 Moshiko LM   : Ready (speech-text model)     ║
║  🖥️  Device       : {DEVICE:<28} ║
╚══════════════════════════════════════════════════╝
""")
print("➡️  Ready for TEST Cells (6, 7, 8, 9)")

---
## 🧪 CELL 6 — TEST 1: Basic Audio Input → Speech Output

In [ ]:
# ============================================================
# CELL 6 — TEST 1: Basic Audio Input → Speech Output
# ============================================================
# PURPOSE: Feed a short audio clip into Moshiko and get
#          synthesized speech back. Saves both input and
#          output to Google Drive for listening/inspection.
#
# 📝 WHAT IT TESTS:
#   - Does the model load and run without errors?
#   - Can it encode audio → tokens → decode back?
#   - Is the output audio intelligible?
# ============================================================

import torch, torchaudio, soundfile as sf
import numpy as np, time, os
from IPython.display import Audio, display
from datetime import datetime

SAMPLE_RATE = 24000   # Mimi operates at 24kHz
TEST_DURATION = 3     # seconds of synthetic test audio

# ---- Step 1: Create a synthetic test audio clip ----
# Using a 440Hz tone (A note) as a simple, reproducible test signal.
# In a real experiment, replace this with actual speech audio.
print("🎵 Creating synthetic test audio (440Hz tone, 3 seconds)...")
t   = torch.linspace(0, TEST_DURATION, SAMPLE_RATE * TEST_DURATION)
wav = (0.3 * torch.sin(2 * np.pi * 440 * t)).unsqueeze(0).unsqueeze(0)  # [1, 1, T]
print(f"  Input shape: {wav.shape} | Sample rate: {SAMPLE_RATE} Hz")

# ---- Step 2: Save input audio to Drive ----
input_path = f"{FOLDERS['audio_in']}/test_input_440hz.wav"
sf.write(input_path, wav.squeeze().numpy(), SAMPLE_RATE)
print(f"  💾 Input saved: {input_path.replace(BASE_DIR, 'Moshiko_Project')}")

# ---- Step 3: Encode with Mimi (audio → tokens) ----
print("\n⚙️  Encoding audio → tokens (Mimi codec)...")
wav_gpu = wav.to(DEVICE)
encode_start = time.time()
with torch.no_grad():
    codes = mimi.encode(wav_gpu)    # [B, K=8, T_codes]
encode_time = time.time() - encode_start
print(f"  ✅ Encoded shape : {codes.shape}  (B × codebooks × time)")
print(f"  ⏱️  Encode time  : {encode_time*1000:.1f} ms")

# ---- Step 4: Decode tokens back to audio ----
print("\n🔊 Decoding tokens → audio...")
decode_start = time.time()
with torch.no_grad():
    decoded_wav = mimi.decode(codes)   # [B, 1, T]
decode_time = time.time() - decode_start
print(f"  ✅ Output shape : {decoded_wav.shape}")
print(f"  ⏱️  Decode time : {decode_time*1000:.1f} ms")

# ---- Step 5: Save output audio to Drive ----
ts = datetime.now().strftime('%Y%m%d_%H%M%S')
output_path = f"{FOLDERS['audio_out']}/test_output_{ts}.wav"
out_audio = decoded_wav.squeeze().cpu().numpy()
sf.write(output_path, out_audio, SAMPLE_RATE)
print(f"  💾 Output saved: {output_path.replace(BASE_DIR, 'Moshiko_Project')}")

# ---- Step 6: Play both in notebook ----
print("\n▶️  INPUT audio:")
display(Audio(wav.squeeze().numpy(), rate=SAMPLE_RATE))
print("▶️  OUTPUT audio (after Mimi encode→decode):")
display(Audio(out_audio, rate=SAMPLE_RATE))

# ---- Step 7: Quality metric — SNR (Signal-to-Noise Ratio) ----
# Trim to same length for comparison
min_len   = min(wav.squeeze().shape[0], decoded_wav.squeeze().cpu().shape[0])
original  = wav.squeeze().numpy()[:min_len]
reconstructed = out_audio[:min_len]
noise     = original - reconstructed
snr       = 10 * np.log10(np.mean(original**2) / (np.mean(noise**2) + 1e-8))
print(f"\n📊 Quality metric: SNR = {snr:.2f} dB")
print(f"   (Higher is better. >20dB = good reconstruction)")

# ---- Save result summary ----
result = {
    'test': 'Basic Audio Encode-Decode',
    'timestamp': ts,
    'input_shape': str(wav.shape),
    'output_shape': str(decoded_wav.shape),
    'encode_time_ms': round(encode_time * 1000, 2),
    'decode_time_ms': round(decode_time * 1000, 2),
    'snr_db': round(float(snr), 2),
    'input_file': input_path,
    'output_file': output_path,
}
import json
with open(f"{FOLDERS['outputs']}/test1_results_{ts}.json", 'w') as f:
    json.dump(result, f, indent=2)

print(f"""
╔══════════════════════════════════════════════════════╗
║  ✅ TEST 1 COMPLETE                                  ║
║  🎵 Audio encoded & decoded successfully             ║
║  📊 SNR: {snr:.2f} dB                                   ║
║  ⏱️  Encode: {encode_time*1000:.0f}ms | Decode: {decode_time*1000:.0f}ms               ║
║  💾 Results saved to Drive                           ║
╚══════════════════════════════════════════════════════╝
""")

---
## ⏱️ CELL 7 — TEST 2: Latency & Speed Benchmark

In [ ]:
# ============================================================
# CELL 7 — TEST 2: Latency & Speed Benchmark
# ============================================================
# PURPOSE: Measure how fast Mimi encodes/decodes audio at
#          different input lengths. This is important for
#          your university project — shows real-world latency
#          and compares to Moshi's theoretical 200ms target.
#
# 📝 WHAT IT MEASURES:
#   - Encode latency (ms) for 1s, 3s, 5s, 10s audio
#   - Decode latency (ms)
#   - Real-time factor (RTF): <1.0 means faster than real-time
#   - GPU memory usage
# ============================================================

import torch, numpy as np, json, time
import matplotlib.pyplot as plt
from datetime import datetime
from IPython.display import display

SAMPLE_RATE = 24000
WARMUP_RUNS = 3
BENCH_RUNS  = 5    # average over this many runs
DURATIONS   = [1, 3, 5, 10]   # seconds of audio to test

print("⏱️  Starting latency benchmark...")
print(f"   Warmup: {WARMUP_RUNS} runs | Benchmark: {BENCH_RUNS} runs each\n")

results = []

for dur in DURATIONS:
    print(f"  Testing {dur}s audio clip...", end='', flush=True)

    # Synthetic audio
    wav = (0.3 * torch.sin(2 * np.pi * 440 * 
           torch.linspace(0, dur, SAMPLE_RATE * dur))
          ).unsqueeze(0).unsqueeze(0).to(DEVICE)

    # Warmup (GPU needs to warm up JIT kernels)
    for _ in range(WARMUP_RUNS):
        with torch.no_grad():
            codes = mimi.encode(wav)
            _     = mimi.decode(codes)
    if DEVICE == 'cuda':
        torch.cuda.synchronize()

    # Benchmark encode
    enc_times = []
    for _ in range(BENCH_RUNS):
        if DEVICE == 'cuda': torch.cuda.synchronize()
        t0 = time.perf_counter()
        with torch.no_grad():
            codes = mimi.encode(wav)
        if DEVICE == 'cuda': torch.cuda.synchronize()
        enc_times.append(time.perf_counter() - t0)

    # Benchmark decode
    dec_times = []
    for _ in range(BENCH_RUNS):
        if DEVICE == 'cuda': torch.cuda.synchronize()
        t0 = time.perf_counter()
        with torch.no_grad():
            _ = mimi.decode(codes)
        if DEVICE == 'cuda': torch.cuda.synchronize()
        dec_times.append(time.perf_counter() - t0)

    enc_mean = np.mean(enc_times) * 1000
    dec_mean = np.mean(dec_times) * 1000
    total_ms = enc_mean + dec_mean
    rtf      = total_ms / (dur * 1000)   # Real-time factor

    mem_mb = torch.cuda.memory_allocated() / 1e6 if DEVICE == 'cuda' else 0

    results.append({
        'duration_s':   dur,
        'encode_ms':    round(enc_mean, 2),
        'decode_ms':    round(dec_mean, 2),
        'total_ms':     round(total_ms, 2),
        'rtf':          round(rtf, 4),
        'gpu_mem_mb':   round(mem_mb, 1),
    })
    print(f" Encode:{enc_mean:.0f}ms | Decode:{dec_mean:.0f}ms | RTF:{rtf:.3f}")

# ---- Print table ----
print("\n📊 BENCHMARK RESULTS")
print(f"  {'Duration':>8} {'Encode(ms)':>12} {'Decode(ms)':>12} {'Total(ms)':>11} {'RTF':>8} {'Status':>10}")
print("  " + "-"*65)
for r in results:
    status = '✅ Faster than RT' if r['rtf'] < 1.0 else '⚠️  Slower than RT'
    print(f"  {r['duration_s']:>7}s {r['encode_ms']:>12.1f} "
          f"{r['decode_ms']:>12.1f} {r['total_ms']:>11.1f} "
          f"{r['rtf']:>8.3f} {status}")
print(f"\n  Note: RTF < 1.0 = faster than real-time (good!)")
print(f"  Moshi's target: 200ms end-to-end")

# ---- Plot results ----
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
fig.suptitle('Moshiko Latency Benchmark (T4 GPU)', fontsize=13, fontweight='bold')

durs = [r['duration_s'] for r in results]
encs = [r['encode_ms']  for r in results]
decs = [r['decode_ms']  for r in results]
rtfs = [r['rtf']        for r in results]

ax1 = axes[0]
x   = np.arange(len(durs))
w   = 0.35
ax1.bar(x - w/2, encs, w, label='Encode', color='#2196F3')
ax1.bar(x + w/2, decs, w, label='Decode', color='#FF9800')
ax1.set_xticks(x)
ax1.set_xticklabels([f'{d}s' for d in durs])
ax1.set_xlabel('Audio Duration')
ax1.set_ylabel('Time (ms)')
ax1.set_title('Encode vs Decode Latency')
ax1.legend()
ax1.grid(axis='y', alpha=0.3)

ax2 = axes[1]
colors = ['#4CAF50' if r < 1.0 else '#F44336' for r in rtfs]
ax2.bar([f'{d}s' for d in durs], rtfs, color=colors)
ax2.axhline(1.0, color='red', linestyle='--', label='Real-time threshold (RTF=1)')
ax2.set_xlabel('Audio Duration')
ax2.set_ylabel('Real-Time Factor (RTF)')
ax2.set_title('Real-Time Factor (lower = faster)')
ax2.legend()
ax2.grid(axis='y', alpha=0.3)

plt.tight_layout()
ts = datetime.now().strftime('%Y%m%d_%H%M%S')
plot_path = f"{FOLDERS['benchmarks']}/latency_benchmark_{ts}.png"
plt.savefig(plot_path, dpi=150, bbox_inches='tight')
plt.show()
print(f"  💾 Plot saved: {plot_path.replace(BASE_DIR, 'Moshiko_Project')}")

# ---- Save JSON results ----
json_path = f"{FOLDERS['benchmarks']}/latency_results_{ts}.json"
with open(json_path, 'w') as f:
    json.dump({'benchmark': results, 'device': DEVICE, 'model': MODEL_REPO}, f, indent=2)
print(f"  💾 Data saved : {json_path.replace(BASE_DIR, 'Moshiko_Project')}")

print("""
╔══════════════════════════════════════════════╗
║  ✅ TEST 2 COMPLETE — Benchmark done!        ║
║  📊 Plot & JSON saved to Drive               ║
╚══════════════════════════════════════════════╝
""")

---
## 🔬 CELL 8 — TEST 3: Compare bf16 vs q8 Quality

In [ ]:
# ============================================================
# CELL 8 — TEST 3: BF16 vs Q8 Quality Comparison
# ============================================================
# PURPOSE: Compare audio reconstruction quality between the
#          full-precision (bf16) Mimi codec and the int8 (q8)
#          version. Measures SNR, MSE, and listening quality.
#
# 📝 WHY THIS MATTERS FOR YOUR PROJECT:
#   - Quantization reduces VRAM but may lose quality
#   - This test QUANTIFIES that quality loss
#   - This data can go directly into your university report!
# ============================================================

import torch, numpy as np, soundfile as sf, json
import matplotlib.pyplot as plt, time
from IPython.display import Audio, display
from datetime import datetime

SAMPLE_RATE = 24000

# Check bf16 Mimi file exists
mimi_bf16_path = f"{FOLDERS['models_bf16']}/mimi_weight.pt"
if not os.path.exists(mimi_bf16_path):
    print("⚠️  BF16 Mimi not found. Re-run Cell 4 to download it.")
    raise FileNotFoundError(mimi_bf16_path)

# ---- Load BF16 Mimi ----
from moshi.models import loaders
print("⏳ Loading Mimi BF16 for comparison...")
mimi_bf16 = loaders.get_mimi(mimi_bf16_path, device=DEVICE)
mimi_bf16.set_num_codebooks(8)
mimi_bf16.eval()
print("  ✅ BF16 Mimi loaded")

# ---- Test signals: different audio types ----
test_signals = {
    'Pure tone 440Hz': lambda: 0.3 * torch.sin(2 * np.pi * 440 * 
                               torch.linspace(0, 2, SAMPLE_RATE * 2)),
    'Mixed tones':     lambda: 0.2 * (
                               torch.sin(2 * np.pi * 440 * torch.linspace(0, 2, SAMPLE_RATE*2)) +
                               torch.sin(2 * np.pi * 880 * torch.linspace(0, 2, SAMPLE_RATE*2)) +
                               torch.sin(2 * np.pi * 220 * torch.linspace(0, 2, SAMPLE_RATE*2))),
    'White noise':     lambda: 0.1 * torch.randn(SAMPLE_RATE * 2),
    'Chirp (sweep)':   lambda: 0.3 * torch.sin(2 * np.pi * 
                               (200 + 1800 * torch.linspace(0, 1, SAMPLE_RATE*2)**2) * 
                               torch.linspace(0, 2, SAMPLE_RATE*2)),
}

comparison_results = []

print("\n🔬 Running comparison...\n")
print(f"  {'Signal':<20} {'Q8 SNR':>10} {'BF16 SNR':>10} {'Diff':>8} {'Q8 faster?':>12}")
print("  " + "-"*65)

for signal_name, signal_fn in test_signals.items():
    wav = signal_fn().unsqueeze(0).unsqueeze(0).to(DEVICE)  # [1,1,T]

    # Q8 encode-decode
    t0 = time.perf_counter()
    with torch.no_grad():
        codes_q8 = mimi.encode(wav)
        out_q8   = mimi.decode(codes_q8)
    time_q8 = (time.perf_counter() - t0) * 1000

    # BF16 encode-decode
    t0 = time.perf_counter()
    with torch.no_grad():
        codes_bf16 = mimi_bf16.encode(wav)
        out_bf16   = mimi_bf16.decode(codes_bf16)
    time_bf16 = (time.perf_counter() - t0) * 1000

    # Quality metrics
    def compute_snr(original, reconstructed):
        min_len = min(original.shape[-1], reconstructed.shape[-1])
        orig = original.squeeze().cpu().numpy()[:min_len].astype(np.float32)
        rec  = reconstructed.squeeze().cpu().numpy()[:min_len].astype(np.float32)
        noise = orig - rec
        return 10 * np.log10(np.mean(orig**2) / (np.mean(noise**2) + 1e-8))

    snr_q8   = compute_snr(wav, out_q8)
    snr_bf16 = compute_snr(wav, out_bf16)
    diff     = snr_bf16 - snr_q8
    faster   = '✅ Yes' if time_q8 < time_bf16 else '❌ No'

    comparison_results.append({
        'signal':      signal_name,
        'snr_q8_db':   round(float(snr_q8),   2),
        'snr_bf16_db': round(float(snr_bf16), 2),
        'snr_diff_db': round(float(diff),     2),
        'time_q8_ms':  round(time_q8,  2),
        'time_bf16_ms':round(time_bf16,2),
    })
    print(f"  {signal_name:<20} {snr_q8:>10.2f} {snr_bf16:>10.2f} "
          f"{diff:>+8.2f} {faster:>12}")

# ---- Plot comparison ----
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle('BF16 vs Q8: Mimi Codec Quality Comparison', fontsize=13, fontweight='bold')

names  = [r['signal'] for r in comparison_results]
snr_q8s   = [r['snr_q8_db']   for r in comparison_results]
snr_bf16s = [r['snr_bf16_db'] for r in comparison_results]
x = np.arange(len(names))
w = 0.35

ax1 = axes[0]
ax1.bar(x - w/2, snr_q8s,   w, label='Q8 (INT8)',  color='#FF9800')
ax1.bar(x + w/2, snr_bf16s, w, label='BF16 (full)', color='#2196F3')
ax1.set_xticks(x)
ax1.set_xticklabels(names, rotation=15, ha='right', fontsize=9)
ax1.set_ylabel('SNR (dB)')
ax1.set_title('Reconstruction Quality (SNR)')
ax1.legend()
ax1.grid(axis='y', alpha=0.3)

ax2 = axes[1]
t_q8s   = [r['time_q8_ms']   for r in comparison_results]
t_bf16s = [r['time_bf16_ms'] for r in comparison_results]
ax2.bar(x - w/2, t_q8s,   w, label='Q8 (INT8)',  color='#FF9800')
ax2.bar(x + w/2, t_bf16s, w, label='BF16 (full)', color='#2196F3')
ax2.set_xticks(x)
ax2.set_xticklabels(names, rotation=15, ha='right', fontsize=9)
ax2.set_ylabel('Time (ms)')
ax2.set_title('Inference Speed')
ax2.legend()
ax2.grid(axis='y', alpha=0.3)

plt.tight_layout()
ts = datetime.now().strftime('%Y%m%d_%H%M%S')
plot_path = f"{FOLDERS['comparisons']}/bf16_vs_q8_{ts}.png"
plt.savefig(plot_path, dpi=150, bbox_inches='tight')
plt.show()
print(f"  💾 Plot saved: {plot_path.replace(BASE_DIR, 'Moshiko_Project')}")

json_path = f"{FOLDERS['comparisons']}/bf16_vs_q8_results_{ts}.json"
with open(json_path, 'w') as f:
    json.dump(comparison_results, f, indent=2)
print(f"  💾 Data saved: {json_path.replace(BASE_DIR, 'Moshiko_Project')}")

# Clean up BF16 model to free VRAM
del mimi_bf16
torch.cuda.empty_cache()
print("  🧹 BF16 Mimi unloaded to free VRAM")

print("""
╔══════════════════════════════════════════════════════════╗
║  ✅ TEST 3 COMPLETE — bf16 vs q8 comparison done!        ║
║  📊 Plot & JSON saved to Drive                           ║
║  💡 Use these results in your university report!         ║
╚══════════════════════════════════════════════════════════╝
""")

---
## 💾 CELL 9 — Save All Outputs & Generate Final Report

In [ ]:
# ============================================================
# CELL 9 — Save All Outputs & Generate Final Session Report
# ============================================================
# PURPOSE: Collect all results from this session, generate
#          a readable summary report, and save a notebook
#          backup to Drive.
#
# ✅ Run this at the END of every session before closing!
# ============================================================

import os, json, glob
from datetime import datetime

ts = datetime.now().strftime('%Y%m%d_%H%M%S')
print("💾 Saving session summary to Google Drive...\n")

# ---- Count all saved files ----
def count_files(folder):
    return len([f for f in os.listdir(folder) 
                if os.path.isfile(os.path.join(folder, f))])

audio_inputs  = count_files(FOLDERS['audio_in'])
audio_outputs = count_files(FOLDERS['audio_out'])
benchmarks    = count_files(FOLDERS['benchmarks'])
comparisons   = count_files(FOLDERS['comparisons'])

# ---- Read latest results ----
def read_latest_json(folder):
    files = sorted(glob.glob(f"{folder}/*.json"))
    if files:
        with open(files[-1]) as f:
            return json.load(f)
    return None

bench_data = read_latest_json(FOLDERS['benchmarks'])
comp_data  = read_latest_json(FOLDERS['comparisons'])

# ---- Generate human-readable report ----
report_lines = [
    "=" * 62,
    " MOSHIKO PROJECT — SESSION REPORT",
    f" Generated : {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}",
    "=" * 62,
    "",
    "1. ENVIRONMENT",
    "-" * 40,
]

env_path = f"{FOLDERS['logs']}/environment_info.json"
if os.path.exists(env_path):
    with open(env_path) as f:
        env = json.load(f)
    report_lines += [
        f"  GPU     : {env.get('gpu_name', 'N/A')}",
        f"  VRAM    : {env.get('vram_total_gb', 'N/A')} GB",
        f"  Model   : {env.get('selected_model', 'N/A')}",
        f"  Python  : {env.get('python_version', 'N/A')}",
    ]

report_lines += [
    "",
    "2. FILES IN DRIVE",
    "-" * 40,
    f"  Audio inputs   : {audio_inputs} file(s)",
    f"  Audio outputs  : {audio_outputs} file(s)",
    f"  Benchmark files: {benchmarks} file(s)",
    f"  Comparison files: {comparisons} file(s)",
]

if bench_data and 'benchmark' in bench_data:
    report_lines += ["", "3. LATEST BENCHMARK RESULTS", "-" * 40]
    report_lines.append(f"  {'Duration':>8} {'Encode(ms)':>12} {'Decode(ms)':>12} {'RTF':>8}")
    for r in bench_data['benchmark']:
        report_lines.append(
            f"  {r['duration_s']:>7}s {r['encode_ms']:>12.1f} "
            f"{r['decode_ms']:>12.1f} {r['rtf']:>8.3f}"
        )

if comp_data:
    report_lines += ["", "4. BF16 vs Q8 COMPARISON", "-" * 40]
    report_lines.append(f"  {'Signal':<20} {'Q8 SNR':>10} {'BF16 SNR':>10} {'Diff':>8}")
    for r in comp_data:
        report_lines.append(
            f"  {r['signal']:<20} {r['snr_q8_db']:>10.2f} "
            f"{r['snr_bf16_db']:>10.2f} {r['snr_diff_db']:>+8.2f}"
        )

report_lines += [
    "",
    "5. HOW TO RESUME NEXT SESSION",
    "-" * 40,
    "  1. Open this notebook in Colab",
    "  2. Runtime → Change runtime type → T4 GPU",
    "  3. Run Cell 1 (Drive mount)",
    "  4. Run Cell 2 (GPU check)",
    "  5. Run Cell 3 (Install packages)",
    "  6. Run Cell 4 (will SKIP already-downloaded files)",
    "  7. Run Cell 5 (Load model)",
    "  8. Run any test cell you need",
    "",
    "=" * 62,
]

report_text = "\n".join(report_lines)

# Save report
report_path = f"{FOLDERS['outputs']}/session_report_{ts}.txt"
with open(report_path, 'w') as f:
    f.write(report_text)

# Print to screen
print(report_text)
print(f"\n  💾 Report saved: {report_path.replace(BASE_DIR, 'Moshiko_Project')}")

# Update session log
with open(f"{FOLDERS['logs']}/session_log.txt", 'a') as f:
    f.write(f"\n[{datetime.now().strftime('%Y-%m-%d %H:%M:%S')}] Session completed. Report: session_report_{ts}.txt")

print("""
╔══════════════════════════════════════════════════════════╗
║  ✅ CELL 9 COMPLETE — Session saved!                     ║
║  📁 All results are in: MyDrive/Moshiko_Project/         ║
║  📋 Share the Drive folder with your group members       ║
║                                                          ║
║  🎯 NEXT STEPS FOR YOUR PROJECT:                         ║
║     → Step A: Fine-tuning (add classification layer)     ║
║     → Step B: Quantization analysis (q8 vs bf16 report)  ║
╚══════════════════════════════════════════════════════════╝
""")